# 03 — Pipeline STT

Deepgram, Whisper, AssemblyAI, Soniox, Speechmatics, Cartesia.


## Optional: run in Docker

Uncomment the cell below to launch JupyterLab + EmbeddedServer in a container. It's idempotent and a no-op once you're already inside the container. Container helpers for the TypeScript notebooks are tracked separately — see issue #80 for status.


In [ ]:
// Optional — launch the patter-notebooks Docker stack from this cell.
// Skip this cell to run on your host runtime. Idempotent if uncommented.
//
// import * as _setup from "./_setup.ts";
// await _setup.startDocker();             // build + up -d, prints http://localhost:8888
// await _setup.startDocker({ openUrl: true });  // …and open the browser tab


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


### Local mode
Construct a Patter instance with a Twilio carrier.


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises STT provider construction, audio transcoding, and (T3) live transcription.


### STT provider construction


In [ ]:
import { DeepgramSTT, WhisperSTT, OpenAITranscribeSTT } from "getpatter";
await cell('stt_providers', { tier: 1, env }, () => {
  const dg = new DeepgramSTT({ apiKey: 'dg-test', model: 'nova-2', language: 'en-US' });
  const wh = new WhisperSTT({ apiKey: 'sk-test', model: 'whisper-1' });
  const ot = new OpenAITranscribeSTT({ apiKey: 'sk-test' });
  console.log(`Deepgram: model=${dg.model}  lang=${dg.language}`);
  console.log(`Whisper:  model=${wh.model}`);
  console.log(`OpenAI Transcribe: ${ot.constructor.name}`);
});


### μ-law ↔ PCM-16 transcoding roundtrip


In [ ]:
import { mulawToPcm16, pcm16ToMulaw } from "getpatter";
await cell('mulaw_transcoding', { tier: 1, env }, () => {
  const pcm = new Uint8Array(1600).fill(0);  // 800 silent 16-bit samples
  const mulaw = pcm16ToMulaw(Buffer.from(pcm.buffer));
  const recovered = mulawToPcm16(mulaw);
  console.log(`PCM: ${pcm.length}B  μ-law: ${mulaw.length}B  recovered: ${recovered.length}B`);
});


### 8kHz → 16kHz resampling


In [ ]:
import { resample8kTo16k, resample16kTo8k } from "getpatter";
await cell('resampler', { tier: 1, env }, () => {
  const pcm8k = Buffer.alloc(1600, 0);  // 800 silent samples @ 8kHz
  const pcm16k = resample8kTo16k(pcm8k);
  const back = resample16kTo8k(pcm16k);
  console.log(`8kHz: ${pcm8k.length}B  16kHz: ${pcm16k.length}B  roundtrip: ${back.length}B`);
});


### Live: Deepgram transcription  *(T3 — requires `DEEPGRAM_API_KEY`)*


In [ ]:
await cell('deepgram_live', { tier: 3, required: ['deepgramKey'], env }, async () => {
  const fs = await import('fs/promises');
  const audio = await fs.readFile('./fixtures/audio/hello_world_16khz_pcm.wav');
  const resp = await fetch('https://api.deepgram.com/v1/listen?model=nova-2&language=en-US', {
    method: 'POST',
    headers: { Authorization: `Token ${env.deepgramKey}`, 'Content-Type': 'audio/wav' },
    body: audio,
  });
  const data = await resp.json() as any;
  const transcript = data.results.channels[0].alternatives[0].transcript;
  console.log(`Deepgram transcript: ${JSON.stringify(transcript)}`);
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Calls a real number through the Pipeline engine using Deepgram STT. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
await cell('live_preflight', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'DEEPGRAM_API_KEY'], env }, () => {
  console.log(`  carrier:  Twilio ${env.twilioNumber}  →  ${env.targetNumber}`);
  console.log('  STT:      Deepgram  (nova-2-general)');
  console.log(`  webhook:  ${env.publicWebhookUrl || '(ngrok auto-launch)'}`);
});


### Live Pipeline STT call *(T4)*


In [ ]:
import { Patter, Twilio, DeepgramSTT, OpenAILLM, OpenAITTS } from "getpatter";
await cell('live_stt_call', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'DEEPGRAM_API_KEY', 'OPENAI_API_KEY'], env }, async () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: env.twilioSid, authToken: env.twilioToken }),
    phoneNumber: env.twilioNumber,
    webhookUrl: env.publicWebhookUrl,
  });
  const agent = p.agent({
    systemPrompt: 'Greet the caller and say goodbye.',
    stt: new DeepgramSTT({ apiKey: env.deepgramKey }),
    llm: new OpenAILLM({ apiKey: env.openaiKey, model: 'gpt-4o-mini' }),
    tts: new OpenAITTS({ apiKey: env.openaiKey, voice: 'alloy' }),
  });
  try {
    await p.call(env.targetNumber, { agent, firstMessage: 'Hello from Patter STT demo.', ringTimeout: env.maxCallSeconds });
    console.log('✓ Pipeline STT call completed');
  } finally {
    await hangupLeftoverCalls(env);
  }
});
